# 00_ingesta · Capa BRONCE
**Proyecto Integrador Big Data · Grupo 03 · Retail / Comercio (Walmart Store Sales Forecasting)**

Lee los 4 CSV oficiales y los persiste *tal cual* como tablas Delta de la capa bronce (todo como texto, sin corregir nada).
La tipificación y la limpieza son la capa plata (`01_limpieza`).

- **Fuente oficial:** Kaggle, Walmart Recruiting - Store Sales Forecasting, https://www.kaggle.com/c/walmart-recruiting-store-sales-forecasting/data
- **Fecha de descarga:** _completar (06/09/2026)_
- **Destino:** Unity Catalog, `workspace.default`

In [0]:
RUTA = "/Volumes/workspace/default/proyecto_integrador_grupo03"
ESQUEMA = "workspace.default"
FUENTE_URL = "https://www.kaggle.com/c/walmart-recruiting-store-sales-forecasting/data"
ARCHIVOS = {"train": "train.csv.gz", "test": "test.csv", "features": "features.csv", "stores": "stores.csv"}

## 1. Lectura y persistencia en bronce
Se lee con `inferSchema=False`: todo llega como `string`. Así el nulo `NA` de Kaggle sigue visible en bronce y la
decisión de qué hacer con él queda documentada en plata, no escondida en la lectura.

In [0]:
from pyspark.sql import functions as F

bronce = {}
for nombre, archivo in ARCHIVOS.items():
    df = (spark.read.option("header", True).option("inferSchema", False)
          .csv(f"{RUTA}/{archivo}")
          .withColumn("_fuente_url", F.lit(FUENTE_URL))
          .withColumn("_archivo", F.lit(archivo))
          .withColumn("_ingesta_ts", F.current_timestamp()))
    df.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{ESQUEMA}.bronce_{nombre}")
    bronce[nombre] = spark.table(f"{ESQUEMA}.bronce_{nombre}")
    print(f"bronce_{nombre}: {bronce[nombre].count():,} filas")

bronce_train: 421,570 filas
bronce_test: 115,064 filas
bronce_features: 8,190 filas
bronce_stores: 45 filas


## 2. Validación 1: conteo de filas contra el archivo fuente
Los conteos esperados salen de la exploración local de la Fase 1.

In [0]:
ESPERADO = {"train": 421570, "test": 115064, "features": 8190, "stores": 45}
filas = [(n, bronce[n].count(), ESPERADO[n], bronce[n].count() == ESPERADO[n]) for n in ESPERADO]
display(spark.createDataFrame(filas, ["tabla", "filas_bronce", "filas_esperadas", "coincide"]))

tabla,filas_bronce,filas_esperadas,coincide
train,421570,421570,true
test,115064,115064,true
features,8190,8190,true
stores,45,45,true


## 3. Validación 2: esquema

In [0]:
for n, df in bronce.items():
    print(f"--- bronce_{n}")
    df.printSchema()

--- bronce_train
root
 |-- Store: string (nullable = true)
 |-- Dept: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Weekly_Sales: string (nullable = true)
 |-- IsHoliday: string (nullable = true)
 |-- _fuente_url: string (nullable = true)
 |-- _archivo: string (nullable = true)
 |-- _ingesta_ts: timestamp (nullable = true)

--- bronce_test
root
 |-- Store: string (nullable = true)
 |-- Dept: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- IsHoliday: string (nullable = true)
 |-- _fuente_url: string (nullable = true)
 |-- _archivo: string (nullable = true)
 |-- _ingesta_ts: timestamp (nullable = true)

--- bronce_features
root
 |-- Store: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Temperature: string (nullable = true)
 |-- Fuel_Price: string (nullable = true)
 |-- MarkDown1: string (nullable = true)
 |-- MarkDown2: string (nullable = true)
 |-- MarkDown3: string (nullable = true)
 |-- MarkDown4: string (nullable = true)
 |-- M

## 4. Validación 3: nulos por columna (incluye el literal `NA`)

In [0]:
def nulos_por_columna(df, nombre):
    cols = [c for c in df.columns if not c.startswith("_")]
    agg = [F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == "NA") | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c) for c in cols]
    fila = df.agg(*agg).first().asDict()
    total = df.count()
    return [(nombre, c, int(v), round(100.0 * v / total, 2)) for c, v in fila.items()]

res = []
for n, df in bronce.items():
    res += nulos_por_columna(df, f"bronce_{n}")
display(spark.createDataFrame(res, ["tabla", "columna", "nulos", "pct_nulos"]))

tabla,columna,nulos,pct_nulos
bronce_train,Store,0,0.0
bronce_train,Dept,0,0.0
bronce_train,Date,0,0.0
bronce_train,Weekly_Sales,0,0.0
bronce_train,IsHoliday,0,0.0
bronce_test,Store,0,0.0
bronce_test,Dept,0,0.0
bronce_test,Date,0,0.0
bronce_test,IsHoliday,0,0.0
bronce_features,Store,0,0.0
